# Tiền xử lý bước 0

## Bước này tạo ra file:

- df_inter.parquet: id của người và sản phẩm đã chuyển sang số vẫn lưu 2 cột để map

Vì code cũ tác giả có đánh lable nhưng qua bước 1 thì chuyển xài label khác nên bước này chỉ map id thôi.


# 5-core filtering

- Extracting U-I interactions and performing 5-core, re-indexing
- dataset located at: http://jmcauley.ucsd.edu/data/amazon/links.html, rating only file in "Small" subsets for experimentation


In [52]:
import os
import pandas as pd

In [53]:
PATH = "./data/2014"

In [54]:
df = pd.read_parquet(os.path.join(PATH, "df_rating.parquet"))

In [55]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 160522 entries, 0 to 160521
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userID     160522 non-null  str    
 1   itemID     160522 non-null  str    
 2   rating     160522 non-null  float64
 3   timestamp  160522 non-null  int64  
dtypes: float64(1), int64(1), str(2)
memory usage: 8.5 MB


In [56]:
df.head(3)

,userID,itemID,rating,timestamp
0,A1HK2FQW6KXQB2,097293751X,5.0,1373932800
1,A19K65VY14D13R,097293751X,5.0,1372464000
2,A2LL1TGG90977E,097293751X,5.0,1395187200


## 5-core filtering


In [57]:
print(f"shape: {df.shape}")
df[:5]

shape: (160522, 4)


,userID,itemID,rating,timestamp
0,A1HK2FQW6KXQB2,097293751X,5.0,1373932800
1,A19K65VY14D13R,097293751X,5.0,1372464000
2,A2LL1TGG90977E,097293751X,5.0,1395187200
3,A5G19RYX8599E,097293751X,5.0,1376697600
4,A2496A4EWMLQ7,097293751X,4.0,1396310400


In [58]:
# Bỏ giá trị null với trùng
learner_id, course_id, tmstmp_str = "userID", "itemID", "timestamp"

df.dropna(subset=[learner_id, course_id, tmstmp_str], inplace=True)
df.drop_duplicates(subset=[learner_id, course_id, tmstmp_str], inplace=True)
print(f"After dropped: {df.shape}")
df[:3]

After dropped: (160522, 4)


,userID,itemID,rating,timestamp
0,A1HK2FQW6KXQB2,097293751X,5.0,1373932800
1,A19K65VY14D13R,097293751X,5.0,1372464000
2,A2LL1TGG90977E,097293751X,5.0,1395187200


In [59]:
from collections import Counter
import numpy as np

min_u_num, min_i_num = 5, 5


# Hàm này có nhiệm vụ tìm ra danh sách các ID "không hợp lệ" dựa trên số lượng tương tác.
def get_illegal_ids_by_inter_num(df, field, max_num=None, min_num=None):
    if field is None:
        return set()
    if max_num is None and min_num is None:
        return set()

    max_num = max_num or np.inf
    min_num = min_num or -1

    ids = df[field].values
    inter_num = Counter(ids)
    ids = {
        id_ for id_ in inter_num if inter_num[id_] < min_num or inter_num[id_] > max_num
    }
    print(f"{len(ids)} illegal_ids_by_inter_num, field={field}")

    return ids


# Đây là hàm quan trọng nhất, thực hiện lọc dữ liệu theo vòng lặp cho đến khi mọi User và Item đều đạt chuẩn "5-core".
def filter_by_k_core(df):
    while True:
        ban_users = get_illegal_ids_by_inter_num(
            df, field=learner_id, max_num=None, min_num=min_u_num
        )
        ban_items = get_illegal_ids_by_inter_num(
            df, field=course_id, max_num=None, min_num=min_i_num
        )
        if len(ban_users) == 0 and len(ban_items) == 0:
            return

        dropped_inter = pd.Series(False, index=df.index)
        if learner_id:
            dropped_inter |= df[learner_id].isin(ban_users)
        if course_id:
            dropped_inter |= df[course_id].isin(ban_items)
        print(f"{len(dropped_inter)} dropped interactions")
        df.drop(df.index[dropped_inter], inplace=True)

In [60]:
k_core = 5
filter_by_k_core(df)
print(f"k-core shape: {df.shape}")
print(f"shape after k-core: {df.shape}")
df[:2]

59 illegal_ids_by_inter_num, field=userID
0 illegal_ids_by_inter_num, field=itemID
160522 dropped interactions
0 illegal_ids_by_inter_num, field=userID
8 illegal_ids_by_inter_num, field=itemID
160286 dropped interactions
9 illegal_ids_by_inter_num, field=userID
0 illegal_ids_by_inter_num, field=itemID
160260 dropped interactions
0 illegal_ids_by_inter_num, field=userID
3 illegal_ids_by_inter_num, field=itemID
160225 dropped interactions
4 illegal_ids_by_inter_num, field=userID
0 illegal_ids_by_inter_num, field=itemID
160213 dropped interactions
0 illegal_ids_by_inter_num, field=userID
1 illegal_ids_by_inter_num, field=itemID
160197 dropped interactions
1 illegal_ids_by_inter_num, field=userID
0 illegal_ids_by_inter_num, field=itemID
160193 dropped interactions
0 illegal_ids_by_inter_num, field=userID
0 illegal_ids_by_inter_num, field=itemID
k-core shape: (160189, 4)
shape after k-core: (160189, 4)


,userID,itemID,rating,timestamp
0,A1HK2FQW6KXQB2,097293751X,5.0,1373932800
1,A19K65VY14D13R,097293751X,5.0,1372464000


In [61]:
df["reviewerID"] = df["userID"].copy()
df["asin"] = df["itemID"].copy()

In [62]:
df

,userID,itemID,rating,timestamp,reviewerID,asin
0,A1HK2FQW6KXQB2,097293751X,5.0,1373932800,A1HK2FQW6KXQB2,097293751X
1,A19K65VY14D13R,097293751X,5.0,1372464000,A19K65VY14D13R,097293751X
2,A2LL1TGG90977E,097293751X,5.0,1395187200,A2LL1TGG90977E,097293751X
3,A5G19RYX8599E,097293751X,5.0,1376697600,A5G19RYX8599E,097293751X
4,A2496A4EWMLQ7,097293751X,4.0,1396310400,A2496A4EWMLQ7,097293751X
...,...,...,...,...,...,...
160517,A30J0DKNKCF7SR,B00L13XFIE,5.0,1391126400,A30J0DKNKCF7SR,B00L13XFIE
160518,AG4E44KM93P4L,B00L13XFIE,4.0,1343606400,AG4E44KM93P4L,B00L13XFIE
160519,A2UZUH4QHV4HA1,B00L13XFIE,5.0,1364256000,A2UZUH4QHV4HA1,B00L13XFIE
160520,A2Z26PUQPMT5JV,B00L13XFIE,5.0,1391731200,A2Z26PUQPMT5JV,B00L13XFIE


## Re-index


In [63]:
df.reset_index(drop=True, inplace=True)

In [64]:
learner_id

'userID'

In [65]:
uid_field, iid_field = learner_id, course_id

uni_users = pd.unique(df[uid_field])
uni_items = pd.unique(df[iid_field])

u_id_map = {k: i for i, k in enumerate(uni_users)}
i_id_map = {k: i for i, k in enumerate(uni_items)}

df[uid_field] = df[uid_field].map(u_id_map)
df[iid_field] = df[iid_field].map(i_id_map)
df[uid_field] = df[uid_field].astype(int)
df[iid_field] = df[iid_field].astype(int)

In [66]:
df

,userID,itemID,rating,timestamp,reviewerID,asin
0,0,0,5.0,1373932800,A1HK2FQW6KXQB2,097293751X
1,1,0,5.0,1372464000,A19K65VY14D13R,097293751X
2,2,0,5.0,1395187200,A2LL1TGG90977E,097293751X
3,3,0,5.0,1376697600,A5G19RYX8599E,097293751X
4,4,0,4.0,1396310400,A2496A4EWMLQ7,097293751X
...,...,...,...,...,...,...
160184,462,7024,5.0,1391126400,A30J0DKNKCF7SR,B00L13XFIE
160185,3492,7024,4.0,1343606400,AG4E44KM93P4L,B00L13XFIE
160186,11090,7024,5.0,1364256000,A2UZUH4QHV4HA1,B00L13XFIE
160187,15069,7024,5.0,1391731200,A2Z26PUQPMT5JV,B00L13XFIE


In [67]:
df.to_parquet(os.path.join(PATH, "df_inter.parquet"), index=False)

## Reload


In [68]:
indexed_df = pd.read_parquet(os.path.join(PATH, "df_inter.parquet"))
print(f"shape: {indexed_df.shape}")
indexed_df[:4]

shape: (160189, 6)


,userID,itemID,rating,timestamp,reviewerID,asin
0,0,0,5.0,1373932800,A1HK2FQW6KXQB2,097293751X
1,1,0,5.0,1372464000,A19K65VY14D13R,097293751X
2,2,0,5.0,1395187200,A2LL1TGG90977E,097293751X
3,3,0,5.0,1376697600,A5G19RYX8599E,097293751X


In [69]:
u_uni = indexed_df[learner_id].unique()
c_uni = indexed_df[course_id].unique()

print(f"# of unique learners: {len(u_uni)}")
print(f"# of unique courses: {len(c_uni)}")

print("min/max of unique learners: {0}/{1}".format(min(u_uni), max(u_uni)))
print("min/max of unique courses: {0}/{1}".format(min(c_uni), max(c_uni)))

# of unique learners: 19372
# of unique courses: 7025
min/max of unique learners: 0/19371
min/max of unique courses: 0/7024
